# Homework: Vector Search (2026)


## Setup

We use the ONNX `Embedder` for lightweight embeddings (no PyTorch needed).
The model is `Xenova/all-MiniLM-L6-v2` (384 dimensions, normalized vectors).


In [1]:
import numpy as np
from tqdm.auto import tqdm

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import VectorSearch, Index

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
embedder = Embedder(path="../models/Xenova/all-MiniLM-L6-v2")

## Loading the data


In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (`v[0]`)?

* -0.31
* -0.02
* 0.12
* 0.44


### Q1 Answer

**-0.02**

In [4]:
q1_query = "How does approximate nearest neighbor search work?"
v = embedder.encode(q1_query)
v[0]

np.float64(-0.020582036807885073)

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?

* 0.07
* 0.37
* 0.68
* 0.92


### Q2 Answer

**0.37**

In [5]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(doc for doc in documents if doc["filename"] == target_filename)

doc_embed = embedder.encode(target_doc["content"])
cos_sim = v.dot(doc_embed)
cos_sim

np.float64(0.361070280302606)

## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages:

```python
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
```

We embed every chunk's `content` with `encode_batch`, stack the vectors into a matrix `X`, and score the Q1 query against all chunks:

```python
scores = X.dot(v)
```

Which file does the highest-scoring chunk belong to (its `filename`)?

* `02-vector-search/lessons/03-embeddings-dataset.md`
* `02-vector-search/lessons/06-rag-vector.md`
* `02-vector-search/lessons/07-sqlitesearch-vector.md`
* `02-vector-search/lessons/09-onnx-embedder.md`


### Q3 Answer

**`02-vector-search/lessons/07-sqlitesearch-vector.md`**

In [6]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [7]:
chunk_texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(chunk_texts)
X = np.array(X)
X.shape

(295, 384)

In [8]:
scores = X.dot(v)
best_idx = np.argmax(scores)
scores[best_idx]
chunks[best_idx]["filename"]

np.float64(0.648901732433228)

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

We use `VectorSearch` from minsearch and run a search for the following query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

* `02-vector-search/lessons/04-vector-search.md`
* `04-evaluation/lessons/05-search-metrics.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `05-monitoring/lessons/04-metrics.md`


### Q4 Answer

**`04-evaluation/lessons/05-search-metrics.md`**

In [9]:
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [10]:
q4_query = "What metric do we use to evaluate a search engine?"
q4_vec = embedder.encode(q4_query)
q4_results = vindex.search(q4_vec, num_results=5)

for r in q4_results:
    print(r["filename"])

q4_results[0]["filename"]

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md


'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Index the same chunks with `Index` from minsearch. Use `content` as a text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

* `02-vector-search/lessons/01-intro.md`
* `02-vector-search/lessons/02-embeddings.md`
* `02-vector-search/lessons/08-pgvector.md`
* `03-orchestration/lessons/05-rag.md`


### Q5 Answer

**`02-vector-search/lessons/08-pgvector.md`**

In [11]:
q5_query = "How do I store vectors in PostgreSQL?"
q5_vec = embedder.encode(q5_query)

# Vector search
q5_vector_results = vindex.search(q5_vec, num_results=5)
vector_filenames = [r["filename"] for r in q5_vector_results]
print("Vector search top 5:")
for f in vector_filenames:
    print(" ", f)

Vector search top 5:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md


In [12]:
# Text search
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

q5_text_results = index.search(q5_query, num_results=5)
text_filenames = [r["filename"] for r in q5_text_results]
print("Text search top 5:")
for f in text_filenames:
    print(" ", f)

Text search top 5:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md


In [13]:
# Find files in vector results but not in text results
only_in_vector = set(vector_filenames) - set(text_filenames)
only_in_vector

{'02-vector-search/lessons/08-pgvector.md'}

## Q6. Hybrid search

We use Reciprocal Rank Fusion (RRF) to combine vector and text search results.

```python
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
```

Run the query `"How do I give the model access to tools?"` with vector and text search, then fuse the results with `rrf`. Which file is ranked first after RRF?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `01-agentic-rag/lessons/16-other-frameworks.md`


### Q6 Answer

**`01-agentic-rag/lessons/13-function-calling.md`**

In [14]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [15]:
q6_query = "How do I give the model access to tools?"
q6_vec = embedder.encode(q6_query)

q6_vector_results = vindex.search(q6_vec, num_results=5)
q6_text_results = index.search(q6_query, num_results=5)

print("Vector results:")
for r in q6_vector_results:
    print(" ", r["filename"])

print("\nText results:")
for r in q6_text_results:
    print(" ", r["filename"])

Vector results:
  01-agentic-rag/lessons/01-intro.md
  04-evaluation/lessons/02-ground-truth.md
  01-agentic-rag/lessons/16-other-frameworks.md
  01-agentic-rag/lessons/15-frameworks.md
  01-agentic-rag/lessons/13-function-calling.md

Text results:
  01-agentic-rag/lessons/14-agentic-loop.md
  01-agentic-rag/lessons/13-function-calling.md
  01-agentic-rag/lessons/13-function-calling.md
  01-agentic-rag/lessons/13-function-calling.md
  04-evaluation/lessons/02-ground-truth.md


In [16]:
q6_results = rrf([q6_vector_results, q6_text_results])

print("RRF results:")
for r in q6_results:
    print(" ", r["filename"])

q6_results[0]["filename"]

RRF results:
  01-agentic-rag/lessons/13-function-calling.md
  01-agentic-rag/lessons/01-intro.md
  01-agentic-rag/lessons/14-agentic-loop.md
  04-evaluation/lessons/02-ground-truth.md
  01-agentic-rag/lessons/16-other-frameworks.md


'01-agentic-rag/lessons/13-function-calling.md'